### Splitting and Audio Data Augmentation

1. **Split dataset** into **train (70%)**, **validation (15%)**, and **test (15%)** sets (stratified by speaker).
2. **Apply audiomentations** to generate **3× augmented copies of the training split only** (NOT test or val).
3. Save the full dataset into the deliverable folder **`clean_augmented/`**.


In [5]:
!pip install -q audiomentations
import os
import glob
from collections import Counter

src_dir = '/content/cleaned_dataset'
output_dir = '/content/clean_augmented'

# Scan all WAV files recursively
all_files = glob.glob(os.path.join(src_dir, '**', '*.wav'), recursive=True)

# Extract labels based on parent directory name (speaker ID)
files_with_labels = [(f, os.path.basename(os.path.dirname(f))) for f in all_files]

if not files_with_labels:
    print("No audio files found in /content/cleaned_dataset. Check the path.")
else:
    filepaths, labels = zip(*files_with_labels)
    print(f"Total cleaned audio files found: {len(filepaths)}")
    print(f"Number of speakers: {len(set(labels))}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 13.6 MB/s eta 0:00:00
Total cleaned audio files found: 2511
Number of speakers: 50


In [ ]:
from sklearn.model_selection import train_test_split

# 70% Train, 15% Val, 15% Test 
train_files, val_test_files, train_labels, val_test_labels = train_test_split(
    filepaths, labels, test_size=0.30, random_state=42, stratify=labels
)

val_files, test_files, val_labels, test_labels = train_test_split(
    val_test_files, val_test_labels, test_size=0.50, random_state=42, stratify=val_test_labels
)

print(f"Train size : {len(train_files)} (70%)")
print(f"Val size   : {len(val_files)} (15%)")
print(f"Test size  : {len(test_files)} (15%)")


Train size : 1757 (70%)
Val size   : 377 (15%)
Test size  : 377 (15%)


In [7]:
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# Create base directories inside clean_augmented/
all_classes = set(labels)
for split in ['train', 'val', 'test']:
    for cls in all_classes:
        os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)

def copy_file(src, split):
    cls = os.path.basename(os.path.dirname(src))
    dest = os.path.join(output_dir, split, cls, os.path.basename(src))
    try:
        os.link(src, dest)  # Instant hardlink
    except OSError:
        shutil.copy2(src, dest)

# Link clean unaugmented files for train, val, and test
print("Placing unaugmented files into clean_augmented/...")
with ThreadPoolExecutor() as executor:
    list(executor.map(lambda f: copy_file(f, 'train'), train_files))
    list(executor.map(lambda f: copy_file(f, 'val'), val_files))
    list(executor.map(lambda f: copy_file(f, 'test'), test_files))

print("Unaugmented train, val, and test files set up successfully.")


Placing unaugmented files into clean_augmented/...
Unaugmented train, val, and test files set up successfully.


In [ ]:
import torch
import torchaudio
import soundfile as sf
import numpy as np
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift, Shift
from tqdm import tqdm

# 3 Distinct Fast Augmentation Pipelines
aug_pipeline_1 = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=1.0)
])

aug_pipeline_2 = Compose([
    TimeStretch(min_rate=0.85, max_rate=1.15, p=1.0, leave_length_unchanged=False)
])

aug_pipeline_3 = Compose([
    PitchShift(min_semitones=-2, max_semitones=2, p=1.0),
    Shift(min_shift=-0.2, max_shift=0.2, p=0.8)
])

pipelines = [
    ("aug_noise", aug_pipeline_1),
    ("aug_stretch", aug_pipeline_2),
    ("aug_pitch", aug_pipeline_3)
]

print(f"Generating 3x augmented copies for {len(train_files)} training files ONLY...")

augmented_count = 0
for fpath in tqdm(train_files):
    cls = os.path.basename(os.path.dirname(fpath))
    fname_stem = os.path.splitext(os.path.basename(fpath))[0]
    dest_dir = os.path.join(output_dir, 'train', cls)
    
    # Fast load
    y, sr = sf.read(fpath, dtype='float32')
    if y.ndim > 1:
        y = np.mean(y, axis=1)
    
    for tag, pipeline in pipelines:
        y_aug = pipeline(samples=y, sample_rate=sr)
        aug_filename = f"{fname_stem}_{tag}.wav"
        aug_dest = os.path.join(dest_dir, aug_filename)
        sf.write(aug_dest, y_aug, sr)
        augmented_count += 1

total_train = len(train_files) + augmented_count
print(f"\nAugmentation complete!")
print(f"Training set: {len(train_files)} original + {augmented_count} augmented = {total_train} total files.")
print(f"Val set     : {len(val_files)} files.")
print(f"Test set    : {len(test_files)} files.")


Generating 3x augmented copies for 1757 training files ONLY...


100%|██████████| 1757/1757 [20:52<00:00,  1.40it/s]


Augmentation complete!
Training set: 1757 original + 5271 augmented = 7028 total files (4x train size).
Val set     : 377 files (untouched).
Test set    : 377 files (untouched).
